### RAG data ingestion and vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/home/ugi/Playground/rag/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f'length of pdf is {len(pdf_files)}')
    try:
        for pdf_file in pdf_files:
            print("processing pdf files..........")
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['file_name']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f'{pdf_file.name} is being processed and pages are {len(documents)}')
    except Exception as e:
        print(f" x {e}")

    print(f"length of {pdf_file.name} is {len(all_documents)}")
    return all_documents


all_pdf_files = process_all_pdfs("../data")


length of pdf is 8
processing pdf files..........
Meghna Cloud All Services User Manual.pdf is being processed and pages are 184
processing pdf files..........
vaas.pdf is being processed and pages are 1
processing pdf files..........
ezio.pdf is being processed and pages are 1
processing pdf files..........
ACFrOgDofyku2-AH39O-uyFUoN7r7yRM4nHjBz_ROdbZmO-RI-hgF3euFtXk06jOOGkxAymhgHJBIQmkInbk507i3HVnK-O9OnHpXIxpGPOTlGSQu6P4S2Qxq9rXwzSfIGI_9nkUoYVuN9CZV5bi9XFcv2ldSDt_NGRF7wVLrQ==.pdf is being processed and pages are 1
processing pdf files..........
embedding.pdf is being processed and pages are 23
processing pdf files..........
55_Calendar_2026.pdf is being processed and pages are 1
processing pdf files..........
batman.pdf is being processed and pages are 1
processing pdf files..........
Attention.pdf is being processed and pages are 11
length of Attention.pdf is 223


In [3]:
all_pdf_files

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdfs/Meghna Cloud All Services User Manual.pdf', 'total_pages': 184, 'page': 0, 'page_label': '1', 'file_name': 'Meghna Cloud All Services User Manual.pdf', 'file_type': 'pdf'}, page_content='Meghna Cloud Platform User Manual \nPage 1 of 184 \n \nMeghna Cloud All Services User Manual \n \nVersion Number: 1.0.5 \nLast Updated: 1st February 2024 \nTable of Content \n1. What is Meghna Cloud? ..................................................................................................................... 6 \n1.1 Getting to Know Meghna Cloud .................................................................................................. 6 \n1.2 Diverse Service Offerings for Varied Needs ............................................................................... 6 \n1.3 State-of-the-Art Datacenter at Gazipur ............................................................................

In [4]:
#textsplitter
def split_document(document, chunk_size=1000, chunk_overlap=200):
    """Split document into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

In [5]:
chunks = split_document(all_pdf_files)

Split 223 documents into 519 chunks
Content: Meghna Cloud Platform User Manual 
Page 1 of 184 
 
Meghna Cloud All Services User Manual 
 
Version Number: 1.0.5 
Last Updated: 1st February 2024 
Table of Content 
1. What is Meghna Cloud? ........
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdfs/Meghna Cloud All Services User Manual.pdf', 'total_pages': 184, 'page': 0, 'page_label': '1', 'file_name': 'Meghna Cloud All Services User Manual.pdf', 'file_type': 'pdf'}


### Embedding And VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using sentence Transformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
        

    def _load_model(self):
        """load the sentencetransformer model"""
        try:
            print(f"Loading embegging model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]):
        """
        Generate Embedding for a list of texts

        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeeddings wwith shape: {embeddings.shape}")
        return embeddings 
    
    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager
    


Loading embegging model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20119.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully, Embedding dimension: 384


### vector

In [9]:
class VectorStore:
    """Manage document embeddings in a chromadb vector"""
    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """Initialize the vector store 
        Args:
            collection_name: Name of the ChromaDB collection 
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection 
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF Document Embedding for RAG"
                }

            )
            print(f"Vectore store initialsized. Collection: {self.collection_name}")
            print(f"Exisitng Document in Collection: {self.collection_count()}")

        except Exception as e:
            print(f"Eerror initializing vector store: {e}")
            raise

        